# Microsoft Graph API — Webhook Test Notebook
### Sysco RFP AIQ

Run each cell top-to-bottom. Each step shows **PASS / FAIL** with the exact reason.

| Step | What is tested | Needs ngrok? |
|------|---------------|--------------|
| 1 | Authentication (MSAL token) | No |
| 2 | Basic Graph API access | No |
| 3 | Mailbox read permission | No |
| 4 | List existing subscriptions | No |
| 5 | Start local validation server | No |
| 6 | Create webhook subscription | **Yes** |
| 7 | Receive a live notification | **Yes** |
| 8 | Delete subscription (cleanup) | No |

Credentials here match the names in `backend_py/.env`.

In [ ]:
# Install required packages (run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'msal', 'requests', '--quiet'])
print('Dependencies ready')

## Configuration
Copy these values from `backend_py/.env`. Leave `NOTIFICATION_URL` as-is until Step 5.

In [ ]:
# ── Credentials — same names as backend_py/.env ────────────────────────────
MS_TENANT_ID        = 'your-tenant-id'          # MS_TENANT_ID
MS_CLIENT_ID        = 'your-client-id'          # MS_CLIENT_ID
MS_CLIENT_SECRET    = 'your-client-secret'      # MS_CLIENT_SECRET
MS_MAILBOX_USER_ID  = 'COE.BIDS@sysco.com'      # MS_MAILBOX_USER_ID

# Public HTTPS URL for webhook notifications.
# Steps 1-4 run fine without this. For Steps 6-7 start ngrok first:
#   ngrok http 8000
# then paste the https:// URL below.
NOTIFICATION_URL = 'https://your-ngrok-url.ngrok.io'

# ── Sanity check ───────────────────────────────────────────────────────────
missing = [k for k, v in {
    'MS_TENANT_ID': MS_TENANT_ID,
    'MS_CLIENT_ID': MS_CLIENT_ID,
    'MS_CLIENT_SECRET': MS_CLIENT_SECRET,
    'MS_MAILBOX_USER_ID': MS_MAILBOX_USER_ID,
}.items() if 'your-' in v or not v]

if missing:
    print(f'  ⚠   Fill in: {", ".join(missing)}')
else:
    print('  ✅  All credentials set')

In [ ]:
# ── Shared helpers used by every step below ─────────────────────────────────
import msal
import requests
from datetime import datetime, timedelta, timezone
import json as _json

GRAPH_BASE = 'https://graph.microsoft.com/v1.0'
SUBSCRIPTION_ID = None   # filled in at Step 6

def ok(msg):   print(f'  ✅  {msg}')
def fail(msg): print(f'  ❌  {msg}')
def info(msg): print(f'  ℹ   {msg}')
def sep(title): print(f'\n{"-"*50}\n  {title}\n{"-"*50}')

def get_token():
    '''Acquire an app-only access token via MSAL client credentials flow.
    Same auth path as MsGraphEmailService._get_token() in the production code.'''
    app = msal.ConfidentialClientApplication(
        client_id=MS_CLIENT_ID,
        client_credential=MS_CLIENT_SECRET,
        authority=f'https://login.microsoftonline.com/{MS_TENANT_ID}',
    )
    result = app.acquire_token_for_client(scopes=['https://graph.microsoft.com/.default'])
    if 'access_token' not in result:
        raise RuntimeError(result.get('error_description', str(result)))
    return result['access_token']

def auth_headers():
    return {
        'Authorization': f'Bearer {get_token()}',
        'Content-Type': 'application/json',
    }

print('  ✅  Helpers defined')

---
## Step 1 — Authentication
Verifies that the Azure AD app registration credentials are correct and the app can get a token.

In [ ]:
sep('STEP 1: Authentication')
try:
    token = get_token()
    ok('Token acquired successfully')
    info(f'Token preview: {token[:40]}...')
    info(f'Length: {len(token)} chars (JWT)')
except Exception as e:
    fail(f'Authentication failed: {e}')
    print()
    print('  Common causes:')
    print('    AADSTS700016  — CLIENT_ID not found in tenant')
    print('    AADSTS7000215 — CLIENT_SECRET is wrong or expired')
    print('    AADSTS90002   — TENANT_ID is wrong')

---
## Step 2 — Basic Graph API Connectivity
Calls `GET /users/{mailbox}` to confirm the app can reach the Graph API and the mailbox user exists.

In [ ]:
sep('STEP 2: Basic Graph API access')
url = f'{GRAPH_BASE}/users/{MS_MAILBOX_USER_ID}'
try:
    resp = requests.get(url, headers=auth_headers(), timeout=15)
    if resp.status_code == 200:
        u = resp.json()
        ok(f'User found')
        info(f'Display name : {u.get("displayName", "N/A")}')
        info(f'UPN          : {u.get("userPrincipalName", "N/A")}')
        info(f'Mail         : {u.get("mail", "N/A")}')
        info(f'Account type : {u.get("userType", "N/A")}')
    elif resp.status_code == 401:
        fail('Unauthorized — token may have expired, re-run Step 1')
    elif resp.status_code == 403:
        fail('Forbidden — app needs User.Read.All or Directory.Read.All permission')
    elif resp.status_code == 404:
        fail(f'User not found: {MS_MAILBOX_USER_ID!r}')
        info('Check MS_MAILBOX_USER_ID — should be a UPN or object ID')
    else:
        fail(f'HTTP {resp.status_code}: {resp.text[:200]}')
except Exception as e:
    fail(f'Request failed: {e}')

---
## Step 3 — Mailbox Read Access
Reads the inbox to confirm the app has `Mail.Read` (or `Mail.ReadBasic`) application permission with admin consent.

In [ ]:
sep('STEP 3: Mailbox read access')
url = (
    f'{GRAPH_BASE}/users/{MS_MAILBOX_USER_ID}/mailFolders/Inbox/messages'
    '?$filter=isRead eq false'
    '&$top=5'
    '&$select=id,subject,from,receivedDateTime'
)
try:
    resp = requests.get(url, headers=auth_headers(), timeout=15)
    if resp.status_code == 200:
        msgs = resp.json().get('value', [])
        ok(f'Inbox readable — {len(msgs)} unread message(s) returned (max 5)')
        if msgs:
            print()
            for m in msgs:
                date  = m.get('receivedDateTime', '')[:10]
                subj  = m.get('subject', '(no subject)')[:55]
                frm   = m.get('from', {}).get('emailAddress', {}).get('address', '?')
                print(f'    [{date}]  {subj}')
                print(f'             From: {frm}\n')
        else:
            info('Inbox is empty (no unread messages) — polling would return nothing')
    elif resp.status_code == 403:
        fail('Forbidden')
        info('Add "Mail.Read" Application permission in Azure AD and grant admin consent')
    elif resp.status_code == 404:
        fail('Mailbox not found — check MS_MAILBOX_USER_ID')
    else:
        fail(f'HTTP {resp.status_code}: {resp.text[:200]}')
except Exception as e:
    fail(f'Request failed: {e}')

---
## Step 4 — Existing Webhook Subscriptions
Lists any subscriptions already registered for this app. Useful to check for stale subscriptions before creating a new one.

In [ ]:
sep('STEP 4: Existing webhook subscriptions')
url = f'{GRAPH_BASE}/subscriptions'
try:
    resp = requests.get(url, headers=auth_headers(), timeout=15)
    if resp.status_code == 200:
        subs = resp.json().get('value', [])
        if subs:
            ok(f'Found {len(subs)} existing subscription(s):')
            print()
            for s in subs:
                expiry    = s.get('expirationDateTime', '')[:19]
                notif_url = s.get('notificationUrl', '?')
                resource  = s.get('resource', '?')
                sid       = s.get('id', '?')
                print(f'    ID          : {sid}')
                print(f'    Resource    : {resource}')
                print(f'    Notify URL  : {notif_url}')
                print(f'    Expires     : {expiry}')
                print()
        else:
            ok('No active subscriptions — clean slate')
    elif resp.status_code == 403:
        fail('Forbidden — app may need additional delegated permissions to list subscriptions')
    else:
        fail(f'HTTP {resp.status_code}: {resp.text[:200]}')
except Exception as e:
    fail(f'Request failed: {e}')

---
## Step 5 — Start Local Validation Server

This cell starts a lightweight HTTP server on **port 8000** in a background thread.
It does two things:
1. **Validation handshake** — when Graph registers a subscription it sends a `validationToken` query param. The server must echo it back as `text/plain` within **10 seconds**.
2. **Capture notifications** — real email arrival notifications are stored in `received_notifications[]`.

**⚠ Graph cannot reach `localhost`.**  
After this cell runs, open a new terminal and start ngrok:
```
ngrok http 8000
```
Copy the `https://...ngrok.io` URL and paste it into `NOTIFICATION_URL` in the Config cell above, then re-run the Config cell.

In [ ]:
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse, parse_qs

received_notifications = []   # all notifications land here
SERVER_PORT = 8000

class GraphWebhookHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        parsed = urlparse(self.path)
        params = parse_qs(parsed.query)

        # ── Validation handshake (must respond within 10 seconds) ──────────
        if 'validationToken' in params:
            token = params['validationToken'][0]
            self.send_response(200)
            self.send_header('Content-Type', 'text/plain')
            self.end_headers()
            self.wfile.write(token.encode())
            print(f'  ✅  Validation handshake completed — token echoed back')
            print(f'      Token (first 40 chars): {token[:40]}...')
            return

        # ── Real notification (new email arrived) ──────────────────────────
        length = int(self.headers.get('Content-Length', 0))
        body = self.rfile.read(length).decode('utf-8', errors='replace')
        received_notifications.append(body)

        print(f'\n  📬  Notification #{len(received_notifications)} received!')
        try:
            data = _json.loads(body)
            for notif in data.get('value', []):
                print(f'      Change type : {notif.get("changeType", "?")}')
                print(f'      Resource    : {notif.get("resource", "?")}')
                print(f'      Client state: {notif.get("clientState", "?")}')
        except Exception:
            print(f'      Raw body: {body[:300]}')

        self.send_response(202)   # must return 2xx within 10s or Graph retries
        self.end_headers()

    def log_message(self, fmt, *args):
        pass   # silence default access log


sep('STEP 5: Local validation server')
try:
    _webhook_server = HTTPServer(('0.0.0.0', SERVER_PORT), GraphWebhookHandler)
    _server_thread  = threading.Thread(target=_webhook_server.serve_forever)
    _server_thread.daemon = True
    _server_thread.start()
    ok(f'Server running on http://localhost:{SERVER_PORT}')
    ok('Validation handshake handler is ready')
    print()
    info('Now open a new terminal and run:')
    info('    ngrok http 8000')
    info('Copy the https:// URL and update NOTIFICATION_URL in the Config cell, then re-run it.')
except OSError as e:
    if 'Address already in use' in str(e):
        ok(f'Server already running on port {SERVER_PORT} (from a previous run)')
    else:
        fail(f'Could not start server: {e}')

---
## Step 6 — Create Webhook Subscription

**Before running this cell:**
1. Confirm ngrok is running (`ngrok http 8000`)
2. Confirm `NOTIFICATION_URL` in the Config cell is set to your ngrok `https://` URL
3. Re-run the Config cell after updating the URL

When this call is made, Graph will **immediately** send a validation request to your URL.  
The local server started in Step 5 will catch it and respond automatically.

In [ ]:
sep('STEP 6: Create webhook subscription')

if 'your-ngrok' in NOTIFICATION_URL or not NOTIFICATION_URL.startswith('https://'):
    fail('NOTIFICATION_URL is not set to a real public HTTPS URL')
    info('Start ngrok (ngrok http 8000), copy the https:// URL, update Config cell, re-run it')
    SUBSCRIPTION_ID = None
else:
    expiry = (
        datetime.now(timezone.utc) + timedelta(days=3)
    ).strftime('%Y-%m-%dT%H:%M:%S.0000000Z')

    notify_url = NOTIFICATION_URL.rstrip('/') + '/webhooks/graph'

    payload = {
        'changeType': 'created',
        'notificationUrl': notify_url,
        'resource': f'users/{MS_MAILBOX_USER_ID}/mailFolders/Inbox/messages',
        'expirationDateTime': expiry,
        'clientState': 'sysco-rfpaiq-test',
    }

    info(f'Notification URL  : {notify_url}')
    info(f'Mailbox           : {MS_MAILBOX_USER_ID}')
    info(f'Expiry            : {expiry[:19]}')
    print()

    try:
        resp = requests.post(
            f'{GRAPH_BASE}/subscriptions',
            headers=auth_headers(),
            json=payload,
            timeout=30,   # Graph has 10s to validate + some overhead
        )

        if resp.status_code == 201:
            sub = resp.json()
            SUBSCRIPTION_ID = sub['id']
            ok(f'Subscription created!')
            ok(f'ID      : {SUBSCRIPTION_ID}')
            ok(f'Expires : {sub.get("expirationDateTime", "?")[:19]}')
            info('Webhook is now live — Graph will POST here on every new email')
        else:
            SUBSCRIPTION_ID = None
            err = resp.json().get('error', {})
            fail(f'HTTP {resp.status_code} — {err.get("code","?")}: {err.get("message","?")[:200]}')
            print()

            # Specific help per error
            if 'not accessible' in resp.text:
                info('Graph could not reach your notification URL')
                info('→ Check ngrok is running and NOTIFICATION_URL is the ngrok https:// URL')
            elif 'Subscription validation request failed' in resp.text:
                info('The server did not respond to the validation handshake in time')
                info('→ Make sure Step 5 ran without errors')
                info('→ Make sure the path /webhooks/graph is handled (the server above handles it)')
            elif 'InvalidClientSecret' in resp.text or resp.status_code == 401:
                info('Token/credential error — re-run Steps 1-2')
            elif resp.status_code == 403:
                info('Missing permissions — add Mail.Read Application permission + admin consent')
    except Exception as e:
        SUBSCRIPTION_ID = None
        fail(f'Request exception: {e}')

---
## Step 7 — Receive a Live Notification

Send an email to `MS_MAILBOX_USER_ID` from any account.  
Within a few seconds, Graph should POST a notification to your server.  
This cell polls `received_notifications[]` for 60 seconds waiting for one to arrive.

In [ ]:
import time

sep('STEP 7: Wait for a live notification')

if not SUBSCRIPTION_ID:
    info('Skipped — no active subscription (re-run Step 6 with a valid ngrok URL)')
else:
    ok(f'Subscription {SUBSCRIPTION_ID[:16]}... is active')
    print()
    info(f'Send an email to:  {MS_MAILBOX_USER_ID}')
    info('Waiting up to 60 seconds...')
    print()

    baseline = len(received_notifications)
    for i in range(12):  # 12 x 5s = 60s
        time.sleep(5)
        current = len(received_notifications)
        if current > baseline:
            ok(f'Received {current - baseline} notification(s)!')
            ok('Graph API webhooks are working end-to-end')
            break
        elapsed = (i + 1) * 5
        print(f'    ...{elapsed}s — {current} total notifications so far')
    else:
        info('No notifications received in 60 seconds')
        info('Things to check:')
        info('  1. Did you actually send an email to the mailbox?')
        info('  2. Is ngrok still running? (check the ngrok terminal)')
        info('  3. Is the subscription still valid? (run Step 4 to list)')
        info('  4. Try running Step 6 again to recreate the subscription')

    print(f'\n  Total notifications received this session: {len(received_notifications)}')

---
## Step 8 — Cleanup
Deletes the test subscription. Always run this — stale subscriptions accumulate against the tenant limit.

In [ ]:
sep('STEP 8: Cleanup')

if not SUBSCRIPTION_ID:
    info('Nothing to clean up')
else:
    try:
        resp = requests.delete(
            f'{GRAPH_BASE}/subscriptions/{SUBSCRIPTION_ID}',
            headers=auth_headers(),
            timeout=15,
        )
        if resp.status_code == 204:
            ok(f'Subscription {SUBSCRIPTION_ID[:16]}... deleted')
            SUBSCRIPTION_ID = None
        elif resp.status_code == 404:
            info('Subscription already expired or deleted')
            SUBSCRIPTION_ID = None
        else:
            fail(f'HTTP {resp.status_code}: {resp.text[:200]}')
    except Exception as e:
        fail(f'Delete exception: {e}')

print()
print('  Run Step 4 to confirm no subscriptions remain.')

---
## Error Reference

| Error | Code | Fix |
|-------|------|-----|
| `AADSTS700016` | 401 | CLIENT_ID not found — check MS_CLIENT_ID |
| `AADSTS7000215` | 401 | CLIENT_SECRET wrong or expired — regenerate in Azure AD |
| `AADSTS90002` | 401 | TENANT_ID wrong |
| `InvalidRequest: not accessible` | 400 | Graph can't reach NOTIFICATION_URL — check ngrok |
| `Subscription validation request failed` | 400 | Server didn't respond to handshake in 10s — check Step 5 |
| `Authorization_RequestDenied` | 403 | Missing `Mail.Read` application permission or no admin consent |
| `ExtensionError` | 400 | Notification URL returned non-2xx on validation |

### Required Azure AD Permissions (Application, not Delegated)

| Permission | Why |
|-----------|-----|
| `Mail.Read` | Read inbox messages |
| `User.Read.All` | Resolve mailbox user (Step 2) |
| `Mail.Send` | Send outbound emails |

All must have **Admin consent granted** (not just added).

### Subscription Limits
- Maximum 3 days expiry for mail resources (`users/.../messages`)
- Renew every 48 hours to stay safe
- Tenant default limit: 10,000 subscriptions per app